## Final Model and Kaggle Predictions

In [ ]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# scaling method
from sklearn.preprocessing import RobustScaler

# wrapper method
from xgboost import XGBClassifier

# imputation
from sklearn.impute import KNNImputer

# random over sampler
from imblearn.over_sampling import RandomOverSampler

## Preprocessing and Feature Selection pipeline

In [ ]:
# import datasets
learn = pd.read_csv("Nata_Files/learn.csv", index_col = 0)
predict = pd.read_csv("Nata_Files/predict.csv", index_col = 0)

In [ ]:
# remove duplicates and rows without target in the training set
learn.drop_duplicates(inplace=True)
learn = learn.dropna(axis = 0, subset = ['quality_class'])

In [ ]:
# split datasets into train (X) and target (y)
X = learn.drop('quality_class', axis=1)
target = learn['quality_class']

In [ ]:
ros = RandomOverSampler(random_state=42)
X, target = ros.fit_resample(X, target)

In [ ]:
# add new columns (log transformations for right-skewed features)
X['sugar_content_log'] = np.log(X['sugar_content'])
predict['sugar_content_log'] = np.log(predict['sugar_content'])
X['salt_ratio_log'] = np.log(X['salt_ratio'])
predict['salt_ratio_log'] = np.log(predict['salt_ratio'])
X['baking_duration_log'] = np.log(X['baking_duration'])
predict['baking_duration_log'] = np.log(predict['baking_duration'])
X['preheating_time_log'] = np.log(X['preheating_time'])
predict['preheating_time_log'] = np.log(predict['preheating_time'])
X['vanilla_extract_log'] = np.log(X['vanilla_extract'])
predict['vanilla_extract_log'] = np.log(predict['vanilla_extract'])

# drop the ones that aren't used to train the model
for col in ["notes_baker", "pastry_type"]:
    if col in X.columns:
        X = X.drop(columns=[col])
        print(f"Removed column from X: {col}")
    else:
        print(f"Column '{col}' already erased from X.")

for col in ["notes_baker", "pastry_type"]:
    if col in predict.columns:
        predict = predict.drop(columns=[col])
        print(f"Removed column from predict: {col}")
    else:
        print(f"Column '{col}' already erased from predict.")


Removed column from X: notes_baker
Removed column from X: pastry_type
Removed column from predict: notes_baker
Removed column from predict: pastry_type


In [ ]:
# uniformize categorical values and remove NaNs
target = target[~X['origin'].isnull()]
X = X[~X['origin'].isnull()]
X['origin'] = X['origin'].str.strip().str.lower()
predict['origin'] = predict['origin'].str.strip().str.lower()

In [ ]:
# encode the categorical column origin so it can be imputed
X['origin_encoded'] = X['origin'].map({'lisboa': 0, 'porto': 1})
predict['origin_encoded'] = predict['origin'].map({'lisboa': 0, 'porto': 1})
X = X.drop(columns = "origin", axis = 1)
predict = predict.drop(columns = "origin", axis = 1)

In [ ]:
# impute the missing values using KNN
knn_imputer = KNNImputer(n_neighbors=6, weights='distance')
X_imputed = knn_imputer.fit_transform(X)
predict_imputed = knn_imputer.transform(predict)
X = pd.DataFrame(X_imputed, columns = X.columns, index = X.index)
predict = pd.DataFrame(predict_imputed, columns= predict.columns, index = predict.index)

In [ ]:
# decode origin after imputation
X['origin_encoded'] = (X['origin_encoded'] >= 0.5).astype(int)
predict['origin_encoded'] =  (predict['origin_encoded'] >= 0.5).astype(int)
X['origin'] = X['origin_encoded'].map({0: 'lisboa', 1: 'porto'})
predict['origin'] = predict['origin_encoded'].map({0: 'lisboa', 1: 'porto'})
X = X.drop(columns='origin_encoded', axis = 1)
predict = predict.drop(columns='origin_encoded', axis=1)

In [ ]:
# change datatypes to reduce memory usage and computational cost

datatypes = {'ambient_humidity': 'int8', 'baking_duration': 'int8', 'cooling_period': 'int8', 'cream_fat_content': 'float32',
             'egg_temperature': 'int16', 'egg_yolk_count': 'int8', 'final_temperature': 'int16', 'lemon_zest_ph': 'float16', 
             'origin': 'category', 'oven_temperature': 'int16', 'preheating_time': 'int16', 'salt_ratio': 'float32', 
             'sugar_content': 'float32', 'vanilla_extract': 'float16', 'sugar_content_log': 'float16', 'salt_ratio_log': 'float16', 
             'baking_duration_log': 'float16', 'preheating_time_log': 'float16', 'vanilla_extract_log': 'float16'}

X = X.astype(datatypes)
predict = predict.astype(datatypes)
target = target.astype('category')

In [ ]:
# split the X and y into train and validation subsets
train, val, train_y, val_y = train_test_split(X, target, test_size=0.15, random_state=1, stratify = target)

In [ ]:
# scale datasets
origin_t = train['origin']
origin_v = val['origin']
origin_p = predict['origin']
train = train.drop('origin', axis=1)
val = val.drop('origin', axis=1)
predict = predict.drop('origin', axis=1)

scaler = RobustScaler().fit(train)
# train data
train_scl = scaler.transform(train) 
train = pd.DataFrame(train_scl, columns = train.columns).set_index(train.index)

# validation data
val_scl = scaler.transform(val) 
val = pd.DataFrame(val_scl, columns = val.columns).set_index(val.index)

# prediction data
predict_scl = scaler.transform(predict) 
predict = pd.DataFrame(predict_scl, columns = predict.columns).set_index(predict.index)

train['origin'] = origin_t
val['origin'] = origin_v
predict['origin'] = origin_p

In [ ]:
# encode categorical features and target
train['origin'] = train['origin'].replace('porto', 0).replace('lisboa', 1)
val['origin'] = val['origin'].replace('porto', 0).replace('lisboa', 1)
predict['origin'] = predict['origin'].replace('porto', 0).replace('lisboa', 1)

train_y = train_y.replace('KO', 0).replace('OK', 1)
val_y = val_y.replace('KO', 0).replace('OK', 1)

## Modelling Pipeline

In [ ]:
cols_drop = ['ambient_humidity', 'cream_fat_content', 'lemon_zest_ph', 'oven_temperature', 'vanilla_extract', 
             'preheating_time_log', 'sugar_content_log', 'salt_ratio_log', 'baking_duration']

train = train.drop(columns=cols_drop, axis=1)
val = val.drop(columns=cols_drop, axis=1)
predict = predict.drop(columns=cols_drop, axis=1)

In [ ]:
# fit model
model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=6,
    min_child_weight=5,
    subsample=0.80,
    colsample_bytree=0.7,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist',
    grow_policy='lossguide',
    enable_categorical=True, 
    early_stopping_rounds=150,
    eval_metric='error',
    n_jobs=-1,
    random_state=42
)


model.fit(train, train_y, 
        eval_set = [(val, val_y)],
        verbose = False)

print(model.get_params)

<bound method XGBModel.get_params of XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7, device=None, early_stopping_rounds=150,
              enable_categorical=True, eval_metric='error', feature_types=None,
              feature_weights=None, gamma=0.1, grow_policy='lossguide',
              importance_type=None, interaction_constraints=None,
              learning_rate=0.01, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=-1, num_parallel_tree=None, ...)>


In [ ]:
# check scores
model.score(train, train_y)

0.8839006439742411

## Predictions

In [ ]:
pd.Series(model.predict(predict), index=predict.index)

id
5201    0
5202    0
5203    0
5204    1
5205    1
       ..
6496    1
6497    1
6498    1
6499    1
6500    1
Length: 1300, dtype: int64

In [ ]:
# predict the target for kaggle
encoded = pd.DataFrame(model.predict(predict), index=predict.index, columns=['quality_class'])
predictions = encoded.replace(0, 'KO').replace(1, 'OK')
predictions

,quality_class
id,
5201,KO
5202,KO
5203,KO
5204,OK
5205,OK
...,...
6496,OK
6497,OK
6498,OK


In [ ]:
# export kaggle predictions
predictions.to_csv('kaggle_predictions.csv')